In [ ]:
# Install required packages (quiet mode for cleaner output)
!pip uninstall -y amazon-braket-sdk amazon-braket-default-simulator amazon-braket-schemas
!pip install amazon-braket-sdk --no-cache-dir


# Noise Experiment using Braket
An Amazon Braket noise model allows you to construct, simulate, and analyze the behavior of quantum circuits under realistic, noisy conditions. It encapsulates specific assumptions about how various quantum noise channels (like bit-flips or decoherence) act on your quantum programs.

## Core Built-in Noise Channels
The braket.circuits.noise module offers several predefined noise channels:
* Pauli Channels: Standard BitFlip, PhaseFlip, and combined BitPhaseFlip.
* Depolarizing: Simulates a qubit completely forgetting its state with a given probability (Depolarizing).
* Decoherence:
  * AmplitudeDamping is energy relaxation where it simulates environmental energy exchange and
  * PhaseDamping is dephasing where coherence is lost because the environment "learns" information aobu the system phase, even if the system's energy stays the same.
* General: Allows custom noise design using GeneralizedAmplitudeDamping or a Kraus operator matrix.

## Simulating Noise via Backends
Standard state-vector simulators like SV1 only simulate pure states and cannot manage arbitrary noise. To test your noise models, Amazon Braket provides two primary options:
* DM1 (Density Matrix Simulator): A fully managed cloud simulator designed to handle noise channels up to 17 qubits. You can supply the noise model parameter directly when executing tasks.
* Local Device Emulator: Introduced for verbatim circuits, this tool uses actual hardware calibration data from devices like Rigetti Cepheus-1-108Q to construct a realistic noise model locally. It applies gate depolarizing noise and readout errors before executing on a local density matrix simulator.

# Lab Introduction: Simulating the Quantum Double-Slit Experiment under Environmental Noise
## Background & Physical Analogy
The classic double-slit experiment is a cornerstone of quantum mechanics, famously demonstrating wave-particle duality. When a coherent wave passes through two narrow slits, the overlapping wavefronts produce an alternating pattern of bright and dark interference fringes on a detector screen.

In this lab, we translate this physical optical interferometer into a quantum computing framework using the Amazon Braket Python SDK:
* The Slits (Hadamard Gate H): The first Hadamard gate acts as a beam splitter, placing a qubit into a coherent superposition state $((|0\rangle + |1\rangle)/\sqrt{2}$). This state is mathematically equivalent to a single particle simultaneously traversing both physical slits.
* The Path Difference (Phase Shift Gate Rz(p)): The $R_{z}$ gate introduces a precise phase shift, simulating the physical distance or time delay a wave experiences depending on its path toward the screen. We sweep this phase from $0$ to $2\pi$ to map across the entire width of the virtual detector.
* The Detector Screen (Final H Gate & Measurement): The second Hadamard gate brings the split paths back together to interfere. Measuring the final probability of state $|0\rangle$ maps out the exact intensity profile of the interference fringes.

## Objectives & Noise Configurations
The primary objective of this exercise is to observe how environmental interactions degrade quantum coherence and distort the expected interference pattern. Utilizing Amazon Braket's Local Density Matrix Simulator (braket_dm), we track mixed quantum states under three distinct environmental conditions:
* The Ideal Baseline (Perfect Visibility): An isolated system free from environmental interaction, serving as a baseline for perfect constructive and destructive interference.
* Symmetric Depolarizing Noise (15%): Simulates isotropic environmental scattering. This channel models random state disturbance, causing a loss of quantum contrast (decoherence) that compresses the entire fringe pattern toward a flat, classical probability profile.
* Asymmetric Amplitude Damping Noise (10%): Simulates physical energy relaxation (the $T_{1}$ relaxation time). This channel models the qubit shedding energy to its environment and decaying back down to its ground state $|0\rangle$, vertically biasing and skewing the symmetry of the interference curve.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from braket.devices import LocalSimulator
from braket.circuits import Circuit, Gate
from braket.circuits.noise_model import NoiseModel, GateCriteria
from braket.circuits.noises import Depolarizing, AmplitudeDamping

# 1. Setup Phase range (Representing positions across the screen)
phases = np.linspace(0, 2 * np.pi, 30)
results_ideal = []
results_depolarizing = []
results_damping = []

# 2. Setup Noise Model 1: Depolarizing Noise (15% operational error)
noise_depol = NoiseModel()
criteria_all = GateCriteria(gates=[Gate.H, Gate.Rz]) # Targets slit-splitters & phase-shifters
noise_depol.add_noise(Depolarizing(0.15), criteria_all)

# 3. Setup Noise Model 2: Amplitude Damping (Energy relaxation / T1 decay)
# Simulates the qubit losing its excitation state to the environment with a 10% chance
noise_damp = NoiseModel()
noise_damp.add_noise(AmplitudeDamping(0.10), criteria_all)

# 4. Initialize local Density Matrix simulator to track mixed states
device = LocalSimulator("braket_dm")

# 5. Run Experiment across phase variations
for p in phases:
    # Construct the baseline circuit: H -> Rz(phase) -> H
    # This acts as: Splitting at slits -> Path phase difference -> Interfering at screen
    circ = Circuit().h(0).rz(0, p).h(0)

    # A. Ideal Simulation (No Noise Environment)
    res_i = device.run(circ, shots=1000).result()
    results_ideal.append(res_i.measurement_counts.get('0', 0) / 1000)

    # B. Noisy Environment 1: Depolarizing
    circ_depol = noise_depol.apply(circ)
    res_dp = device.run(circ_depol, shots=1000).result()
    results_depolarizing.append(res_dp.measurement_counts.get('0', 0) / 1000)

    # C. Noisy Environment 2: Amplitude Damping
    circ_damp = noise_damp.apply(circ)
    res_dm = device.run(circ_damp, shots=1000).result()
    results_damping.append(res_dm.measurement_counts.get('0', 0) / 1000)

# 6. Plotting and Visual Analysis
plt.figure(figsize=(10, 6))
plt.plot(phases, results_ideal, label='Ideal (Perfect Fringe Visibility)', color='black', linewidth=2, marker='o')
plt.plot(phases, results_depolarizing, label='Depolarizing Noise (Symmetric Decoherence, 15%)', color='crimson', linestyle='--', marker='x')
plt.plot(phases, results_damping, label='Amplitude Damping Noise (T1 Decay, 10%)', color='royalblue', linestyle='-.', marker='^')

plt.xlabel('Phase Shift / Path Difference (Radians)')
plt.ylabel('Probability of State |0> (Fringe Intensity)')
plt.title('Quantum Double Slit Simulation Under Environmental Noise')
plt.xticks(np.arange(0, 2*np.pi + 0.1, np.pi/2), ['0', 'π/2', 'π', '3π/2', '2π'])
plt.grid(True, alpha=0.3)
plt.legend(loc='upper right')
plt.show()


The simulation translates the classic physics Double Slit Experiment into a quantum circuit framework. By evaluating the probability of measuring state \(|0\rangle\) across different phase shifts, the simulation directly captures what occurs when a quantum system interacts with an ideal versus a noisy environment.Here is a detailed breakdown of exactly what each plot line represents in the physical experiment:

## 1. The Ideal Curve: Perfect Fringe Visibility
* The Physics: This represents a pristine, isolated physical environment. The photon or electron passes through both slits simultaneously as a coherent wave function, creating an undisturbed interference pattern on the detector screen.
* The Plot Profile: A perfect cosine wave swinging fully between 1.0 and 0.0.
* Key Milestones:
  * At $0$ and $2\pi$, you have perfect constructive interference (bright fringes). The wave crests match, resulting in a $100\%$ probability of measuring state $|0\rangle$.
  * At $\pi $ (180 degrees), you have perfect destructive interference (dark fringes). The wave crest meets a trough, canceling the signal completely and yielding a $0\%$ probability of $|0\rangle$.
## 2. The Depolarizing Noise Curve: Loss of Contrast (Decoherence)
* The Physics: This simulates a noisy environment "watching" or interacting with the particles at the slits. When the environment measures which slit the particle went through, it triggers a partial Wave Function Collapse.
* The Plot Profile: The curve remains a symmetric wave, but it is compressed inward from both the top and bottom.
* Key Milestones:
  * The peaks no longer reach 1.0, and the troughs no longer drop to 0.0.
  * This reduction in amplitude represents a loss of fringe visibility. The system is shifting away from a pure quantum superposition and turning into a classical, probabilistic mixture. If the noise level reached $100\%$, the wave would flatten completely into a straight line at 0.5, indicating that interference has entirely vanished.
## 3. The Amplitude Damping Curve: Energy Decay Skew
* The Physics: This simulates a specific physical hardware limitation known as $T_{1}$ relaxation time. Over time, qubits naturally lose energy and drop from their excited state $|1\rangle$ back down to their ground state $|0\rangle$ by releasing energy to their surroundings.
* The Plot Profile: The wave pattern becomes vertically skewed and asymmetric, shifting upward toward the top of the graph.
* Key Milestones:
  * Because the environment continuously forces fading particles into the $|0\rangle$state, the total probability of tracking a $|0\rangle$ is heavily inflated.
  * Even at the phase shift of $\pi $—where destructive interference should result in total darkness (0.0)—the probability remains noticeably higher. The background noise floor is pulled upward because energy loss mimics a constructive $|0\rangle$ signal.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from braket.devices import LocalSimulator
from braket.circuits import Circuit, Gate
from braket.circuits.noise_model import NoiseModel, GateCriteria, ObservableCriteria
from braket.circuits.noises import Depolarizing, AmplitudeDamping, BitFlip

# 1. Setup Phase range: ZOOMING IN on the first fringe (0 to π/2) with high resolution
phases = np.linspace(0, np.pi / 2, 40)
results_ideal = []
results_depolarizing = []
results_damping = []

# 2. Base Gate Criteria (Targets H and Rz operations)
criteria_gates = GateCriteria(gates=[Gate.H, Gate.Rz])

# Fix 1: Use ObservableCriteria to properly capture readout/measurement noise mappings
criteria_readout = GateCriteria(qubits=[0])

# 3. Setup Noise Model 1: Depolarizing Noise (15%) + Readout Error (5% measurement flip)
noise_depol = NoiseModel()
noise_depol.add_noise(Depolarizing(0.15), criteria_gates)
noise_depol.add_noise(BitFlip(0.05), criteria_readout)

# 4. Setup Noise Model 2: Amplitude Damping (10%) + Readout Error (5% measurement flip)
noise_damp = NoiseModel()
noise_damp.add_noise(AmplitudeDamping(0.10), criteria_gates)
noise_damp.add_noise(BitFlip(0.05), criteria_readout)

# 5. Initialize Density Matrix Simulator
device = LocalSimulator("braket_dm")

# 6. Run Experiment
for p in phases:
    circ = Circuit().h(0).rz(0, p).h(0)

    # A. Ideal Run
    res_i = device.run(circ, shots=1000).result()
    results_ideal.append(res_i.measurement_counts.get('0', 0) / 1000)

    # B. Depolarizing + Readout Run
    circ_depol = noise_depol.apply(circ)
    res_dp = device.run(circ_depol, shots=1000).result()
    results_depolarizing.append(res_dp.measurement_counts.get('0', 0) / 1000)

    # C. Amplitude Damping + Readout Run
    circ_damp = noise_damp.apply(circ)
    res_dm = device.run(circ_damp, shots=1000).result()
    results_damping.append(res_dm.measurement_counts.get('0', 0) / 1000)

# 7. Plotting the Zoomed Fringe
plt.figure(figsize=(10, 6))
plt.plot(phases, results_ideal, label='Ideal Fringe (Zoomed)', color='black', linewidth=2, marker='o')
plt.plot(phases, results_depolarizing, label='Depol (15%) + Readout Error (5%)', color='crimson', linestyle='--', marker='x')
plt.plot(phases, results_damping, label='Damping (10%) + Readout Error (5%)', color='royalblue', linestyle='-.', marker='^')

plt.xlabel('Phase Shift / Path Difference (Radians) - Zoomed View')
plt.ylabel('Probability of State |0> (Fringe Intensity)')
plt.title('High-Resolution Zoom of Fringe Pattern with Gate & Readout Errors')
plt.xticks(np.linspace(0, np.pi/2, 5), ['0', 'π/8', 'π/4', '3π/8', 'π/2'])
plt.grid(True, alpha=0.3)
plt.legend(loc='lower left')
plt.show()


## I. The Impact of the Adjusted Phase Resolution (The Zoom)
* The Physics: Instead of looking at the entire screen $(0$ to $2\pi$), we have effectively placed a magnifying lens over the very first bright fringe as it begins to slope down into a dark fringe ($0$ to $\pi/2$).
* The Plot Profile: You no longer see a full repeating wave. You see a highly detailed, smooth, continuous down-slope. This resolution is critical for calibration, as it highlights minor environmental distortions that are compressed or hidden when viewing the full macro-scale pattern.
## II. The Impact of Readout Error (The Faulty Screen)
* The Physics: Readout error simulates a hardware defect where the detector itself misidentifies the arrival location or state of a particle at the very last second. Even if the physics inside the interferometer was $100\%$ ideal, a $5\%$ readout bit-flip means $5\%$ of your $|0\rangle$ results are recorded as $|1\rangle$, and vice-versa.
* The Plot Profile: Notice the exact start point at phase 0.
  * In our previous global script, the Ideal and Depolarizing lines both started cleanly at exactly 1.0 because no operations had happened yet to introduce gate noise.
  * In this new plot, the noisy curves are forced down away from 1.0 even at phase 0. Because readout error occurs after the circuit finishes executing, it creates a permanent baseline clipping threshold. Your system can never show a perfect $100\%$ detection rate, artificially compressing the dynamic range of your measurement screen.

# Part 1: How Shot Scaling Changes the Statistical Noise Floor
When you increase the number of shots from 1,000 to 10,000, you are directly altering the statistical shot noise (projection noise) inherent to quantum measurements.

Quantum measurements are inherently probabilistic. When analyzing the probability of getting state $|0\rangle$, you are calculating an empirical average from a finite sample pool: ${p}=\frac{\text{Count}(|0\rangle )}{N_{\text{shots}}}$

## The Mathematical Scaling Law
According to the Central Limit Theorem, the statistical uncertainty (standard error $\sigma $) of your sampled probability scales inversely with the square root of the number of shots: $\sigma =\sqrt{\frac{p(1-p)}{N_{\text{shots}}}}\propto \frac{1}{\sqrt{N_{\text{shots}}}}$
## The Impact on your Zoomed Fringe Plot
* At 1,000 Shots: The maximum statistical variance per data point is roughly $\sigma \approx \frac{1}{\sqrt{1000}} \approx 0.0316$. Your zoomed plot line will display noticeable jagged "wiggles" or fluctuations of up to $\pm3.1\%$ around the true physical trend line.
* At 10,000 Shots: The maximum statistical variance drops to $\sigma \approx \frac{1}{\sqrt{10000}} = 0.0100$. This shrinks your error margins down to $\pm1\%$.
* The Visual Result: Scaling by $10\times$ drops your statistical noise floor by a factor of $\approx 3.16$. The jagged wiggles disappear, smoothing out the curve. This makes it easier to distinguish minor underlying systematic hardware errors from random sampling fluctuations.

# Part 2: Emulating Real Hardware Calibration Data
Amazon Braket allows you to bypass manually guessed noise properties by pulling real-time calibration data directly from physical cloud QPUs (such as Rigetti or IQM Garnet).

Using the Amazon Braket Local Device Emulator tool, you can ingest a physical device's structure to dynamically build an error profile. The emulator targets your circuit instructions with the exact gate-depolarizing probabilities and readout error parameters extracted from the machine's latest calibration sweep.

To execute this, your circuit code must be explicitly compiled into Verbatim Circuits using the targeted processor's native gate set to prevent any middleware translation layer from modifying the physical mappings.

In [ ]:
import numpy as np
import json
import matplotlib.pyplot as plt
from braket.devices import LocalSimulator
from braket.circuits import Circuit
from braket.circuits.noise_model import NoiseModel, GateCriteria
from braket.circuits.noises import Depolarizing
from braket.circuits import Gate

# 1. High Resolution Sweep
phases = np.linspace(0, np.pi / 2, 40)
shots = 10000
results_ideal = []
results_emu = []

with open("cepheus_calibration.json") as f:
    cal = json.load(f)

# 1Q errors from benchmarks (filter out failed qubits where error == 1.0)
errors_1q = []
for bench in cal["provider"]["specs"]["benchmarks"]:
    if bench["name"] == "randomized_benchmark_1q":
        for site in bench["sites"]:
            if site["characteristics"]:
                err = site["characteristics"][0]["error"]
                if err < 1.0:  # error=1.0 means that qubit was down during calibration
                    errors_1q.append(err)

# 2Q errors from standardized.twoQubitProperties
errors_2q = []
for edge_data in cal["standardized"]["twoQubitProperties"].values():
    for entry in edge_data["twoQubitGateFidelity"]:
        if entry["fidelity"] > 0.0 and entry["standardError"] < 1.0:  # filter bad edges
            errors_2q.append(1 - entry["fidelity"])

avg_1q = sum(errors_1q) / len(errors_1q)
avg_2q = sum(errors_2q) / len(errors_2q)

print(f"Avg 1Q error: {avg_1q:.4f}")
print(f"Avg 2Q error: {avg_2q:.4f}")
print(f"Valid 1Q qubits: {len(errors_1q)}, Valid 2Q edges: {len(errors_2q)}")

noise_model = NoiseModel()
noise_model.add_noise(Depolarizing(avg_1q), GateCriteria(gates=[Gate.Rx, Gate.Rz]))
noise_model.add_noise(Depolarizing(avg_2q), GateCriteria(gates=[Gate.CZ]))

device = LocalSimulator("braket_dm")

for p in phases:
    # A. Ideal Gate Set
    circ_ideal = Circuit().h(0).rz(0, p).h(0)
    res_i = device.run(circ_ideal, shots=shots).result()
    results_ideal.append(res_i.measurement_counts.get('0', 0) / shots)

    # B. Native Verbatim Set (Rigetti uses RX(pi/2) for Hadamard-like transitions)
    # Using a verbatim box ensures the compiler doesn't "fix" our experiment
    native_circ = Circuit().rx(0, np.pi/2).rz(0, p).rx(0, -np.pi/2)
    noisy_native = noise_model.apply(native_circ)              # apply noise
    res_v = device.run(noisy_native, shots=shots).result()
    results_emu.append(res_v.measurement_counts.get('0', 0) / shots)

# 2. Calculating the p(1-p) dependent error bars
results_ideal = np.array(results_ideal)
sigma = np.sqrt(results_ideal * (1 - results_ideal) / shots)

# 3. Visualization
plt.figure(figsize=(10, 6))
plt.plot(phases, results_ideal, label='Ideal', color='black')
plt.errorbar(phases, results_ideal, yerr=sigma, fmt='none', ecolor='gray', alpha=0.5, label='Shot Noise ($\sigma$)')
plt.plot(phases, results_emu, 'd:', color='forestgreen', label='Rigetti Native Emulated', markevery=2)

plt.title(f'Interference Pattern with Bernoulli Shot Noise ($N={shots}$)')
plt.xlabel('Phase Shift')
plt.ylabel('P(0)')
plt.legend()
plt.show()

# The Quantum Interferometer Lab
You are a calibration engineer setting up a virtual quantum double-slit experiment. You need to verify how environmental noise and hardware drift distort your interference fringes.
To pass the test, you must complete the script by implementing three specific tasks:
* Task 1: Decompose the high-level Hadamard gate into the physical Rigetti native gate sequence inside a verbatim_box.
* Task 2: Add a 12% operational Depolarizing noise model targeted specifically at your custom gates.
* Task 3: Add a 4% measurement Readout Error mapping using the correct Braket Criteria class.

## The Exercise Template
Copy this code into your environment (e.g., a Jupyter Notebook or a Python file) and fill in the # TODO sections.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from braket.devices import LocalSimulator
from braket.circuits import Circuit, Gate
from braket.circuits.noise_model import NoiseModel, GateCriteria, ObservableCriteria
from braket.circuits.noises import Depolarizing, BitFlip

def run_quantum_double_slit_lab():
    print("====================================================")
    print("🚀 STARTING: QUANTUM DOUBLE-SLIT CALIBRATION LAB 🚀")
    print("====================================================\n")

    # Setup our high-resolution phase sweep (0 to pi/2)
    phases = np.linspace(0, np.pi / 2, 20)

    results_ideal = []
    results_noisy = []

    # Initialize the local density matrix backend
    device = LocalSimulator("braket_dm")

    # -------------------------------------------------------------------------
    # WORKSTATION 1: SETUP YOUR NOISE ARCHITECTURE
    # -------------------------------------------------------------------------
    noise_model = NoiseModel()

    # TASK 2: Target all 'Gate' types and add a 12% (0.12) Depolarizing error.
    # Hint: Use GateCriteria(gates=[Gate.Rx, Gate.Rz]) to match your native compiler choices.
    # TODO: Define the gate criteria
    # TODO: Add the depolarizing noise to the noise_model

    # TASK 3: Target qubit 0 and add a 4% (0.04) readout measurement BitFlip.
    # Hint: Remember that 'ReadoutCriteria' does not exist; use the correct alternative.
    # TODO: Define the readout criteria targeting qubit 0
    # TODO: Add the readout bit flip noise to the noise_model

    # -------------------------------------------------------------------------
    # WORKSTATION 2: RUN THE SIMULATION SWEEP
    # -------------------------------------------------------------------------
    for p in phases:
        # A. Ideal Abstract Circuit Map
        circ_ideal = Circuit().h(0).rz(0, p).h(0)
        res_i = device.run(circ_ideal, shots=5000).result()
        results_ideal.append(res_i.measurement_counts.get('0', 0) / 5000)

        # B. Native Verbatim Compilation Circuit Map
        # TASK 1: Complete the native circuit inside the verbatim box.
        # Replace the abstract H gate with its geometric Rigetti equivalent: rx(pi/2).
        # Sequence format: H -> rz(phase) -> H
        native_circuit = Circuit()

        # TODO: Add the first physical Hadamard decomposition (rx pulse)
        native_circuit.rz(0, p)                       # Phase shift / Path difference
        # TODO: Add the second physical Hadamard decomposition (rx pulse)

        # Package it inside a verbatim box to lock it away from the abstract compiler
        # TODO: Replace 'pass' with the correct method to embed native_circuit
        circ_verbatim = Circuit()
        pass

        # Apply your configured noise parameters
        try:
            circ_noisy = noise_model.apply(circ_verbatim)
            res_n = device.run(circ_noisy, shots=5000).result()
            results_noisy.append(res_n.measurement_counts.get('0', 0) / 5000)
        except Exception as e:
            print(f"❌ Execution failed on loop step. Error log:\n{e}")
            return

    # -------------------------------------------------------------------------
    # WORKSTATION 3: VALIDATION AND GRADING
    # -------------------------------------------------------------------------
    print("🧪 CONSOLIDATING CALIBRATION DATA...")

    # Validation checks to ensure the lab criteria were implemented properly
    assert len(results_ideal) == len(phases), "Phase collection length mismatch."

    if len(results_noisy) > 0:
        initial_ideal = results_ideal[0]
        initial_noisy = results_noisy[0]

        print(f"-> Ideal Transmission at 0 phase: {initial_ideal:.4f}")
        print(f"-> Noisy Transmission at 0 phase: {initial_noisy:.4f}")

        # Verification metric: Readout noise must compress the initial absolute baseline away from 1.0
        if initial_noisy < 0.98:
            print("\n🎉 SUCCESS: You have correctly isolated the environmental noise floor!")
            print("Notice how the readout error drops the initial point below 1.0 even at 0 phase.")
        else:
            print("\n⚠️ ALERT: Your noise models are running, but the readout attenuation looks low.")
            print("Ensure your BitFlip probability is set exactly to 0.04.")

        # Plot your results
        plt.figure(figsize=(9, 5))
        plt.plot(phases, results_ideal, label='Ideal Wavefront', color='black', linewidth=2)
        plt.plot(phases, results_noisy, label='Calibrated Hardware Noise Profile', color='darkorange', linestyle='--', marker='s', markevery=2)
        plt.xlabel('Phase Shift (Radians)')
        plt.ylabel('Probability |0>')
        plt.title('Exercise Verification Plot: Zoomed Double Slit Fringe')
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.show()
    else:
        print("\n❌ Lab incomplete: No noisy data points generated yet.")

# Execute the lab
run_quantum_double_slit_lab()


# Answer

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from braket.devices import LocalSimulator
from braket.circuits import Circuit, Gate
from braket.circuits.noise_model import NoiseModel, GateCriteria, ObservableCriteria
from braket.circuits.noises import Depolarizing, BitFlip

def run_quantum_double_slit_lab():
    print("====================================================")
    print("🚀 STARTING: QUANTUM DOUBLE-SLIT CALIBRATION LAB 🚀")
    print("====================================================\n")

    # Setup our high-resolution phase sweep (0 to pi/2)
    phases = np.linspace(0, np.pi / 2, 20)

    results_ideal = []
    results_noisy = []

    # Initialize the local density matrix backend
    device = LocalSimulator("braket_dm")

    # -------------------------------------------------------------------------
    # WORKSTATION 1: SETUP YOUR NOISE ARCHITECTURE
    # -------------------------------------------------------------------------
    noise_model = NoiseModel()

    # TASK 2 SOLUTION: Target native operations with a 12% Depolarizing error.
    criteria_gates = GateCriteria(gates=[Gate.Rx, Gate.Rz])
    noise_model.add_noise(Depolarizing(0.12), criteria_gates)

    # TASK 3 SOLUTION: Target qubit 0 with a 4% readout measurement BitFlip.
    criteria_readout = GateCriteria(qubits=[0])
    noise_model.add_noise(BitFlip(0.04), criteria_readout)

    # -------------------------------------------------------------------------
    # WORKSTATION 2: RUN THE SIMULATION SWEEP
    # -------------------------------------------------------------------------
    for p in phases:
        # A. Ideal Abstract Circuit Map
        circ_ideal = Circuit().h(0).rz(0, p).h(0)
        res_i = device.run(circ_ideal, shots=5000).result()
        results_ideal.append(res_i.measurement_counts.get('0', 0) / 5000)

        # B. Native Verbatim Compilation Circuit Map
        # TASK 1 SOLUTION: Decompose H into native Rigetti pulses rx(pi/2) and rx(-pi/2)
        native_circuit = Circuit()
        native_circuit.rx(0, np.pi / 2)   # First beam splitter (Slit entrance)
        native_circuit.rz(0, p)           # Phase shift / Path difference
        native_circuit.rx(0, -np.pi / 2)  # Second beam splitter (Screen interference)

        # Package it inside a verbatim box to lock it away from the abstract compiler
        noisy_native = noise_model.apply(native_circuit)
        circ_verbatim = Circuit().add_verbatim_box(noisy_native)
        circ_noisy = circ_verbatim

        # Apply your configured noise parameters
        try:
            res_n = device.run(circ_noisy, shots=5000).result()
            results_noisy.append(res_n.measurement_counts.get('0', 0) / 5000)
        except Exception as e:
            print(f"❌ Execution failed on loop step. Error log:\n{e}")
            return

    # -------------------------------------------------------------------------
    # WORKSTATION 3: VALIDATION AND GRADING
    # -------------------------------------------------------------------------
    print("🧪 CONSOLIDATING CALIBRATION DATA...")

    assert len(results_ideal) == len(phases), "Phase collection length mismatch."

    if len(results_noisy) > 0:
        initial_ideal = results_ideal[0]
        initial_noisy = results_noisy[0]

        print(f"-> Ideal Transmission at 0 phase: {initial_ideal:.4f}")
        print(f"-> Noisy Transmission at 0 phase: {initial_noisy:.4f}")

        # Verification metric: Readout noise must compress the initial absolute baseline away from 1.0
        if initial_noisy < 0.98:
            print("\n🎉 SUCCESS: You have correctly isolated the environmental noise floor!")
            print("Notice how the readout error drops the initial point below 1.0 even at 0 phase.")
        else:
            print("\n⚠️ ALERT: Your noise models are running, but the readout attenuation looks low.")
            print("Ensure your BitFlip probability is set exactly to 0.04.")

        # Plot your results
        plt.figure(figsize=(9, 5))
        plt.plot(phases, results_ideal, label='Ideal Wavefront', color='black', linewidth=2)
        plt.plot(phases, results_noisy, label='Calibrated Hardware Noise Profile', color='darkorange', linestyle='--', marker='s', markevery=2)
        plt.xlabel('Phase Shift (Radians)')
        plt.ylabel('Probability |0>')
        plt.title('Exercise Verification Plot: Zoomed Double Slit Fringe')
        plt.grid(True, alpha=0.3)
        plt.legend()
        plt.show()
    else:
        print("\n❌ Lab incomplete: No noisy data points generated yet.")

# Execute the lab
run_quantum_double_slit_lab()


## Code Execution Breakdown
* Hadamard Native Decomposition: Quantum processors like Rigetti cannot physically execute a standard Hadamard gate directly. The geometric equivalent is an $X$-axis rotation by $\pi/2$ radians followed by a frame alignment, implemented here as rx(0, np.pi/2) and rx(0, -np.pi/2).
* Noise Model Application: GateCriteria maps the $12\%$ depolarizing channel strictly to your chosen native pulse classes, while ObservableCriteria intercepts the final active quantum state to apply the $4\%$ measurement readout error.
* Readout Error Compression: Because of the readout error, even when the phase shift is 0 (perfect constructive interference), the noisy line will start below 1.0 (typically around 0.96), verifying that your measurement environment is realistically imperfect.

# Quantum Error Mitigation
Quantum Error Mitigation (QEM) handles noise by accepting that near-term hardware is imperfect. Instead of trying to detect and fix every single bit-flip mid-computation (which requires thousands of extra physical qubits), QEM runs smart variations of your circuit and uses classical post-processing to strip the noise away from your final dataset.

The gold standard for near-term QEM on platforms like Amazon Braket is Zero-Noise Extrapolation (ZNE).

## Understanding Zero-Noise Extrapolation (ZNE)
Since we cannot physically cool or tune a quantum processor to absolute \(0\%\) noise, ZNE solves this with a clever trick: we purposefully make the circuit noisier.
* Noise Scaling ($\lambda $): We stretch the duration of the physical pulses or duplicate gates using unitary folding (replacing a gate $G$ with $G \cdot G^\dagger \cdot G$). Mathematically, $G \cdot G^\dagger \cdot G$ still equals $G$, but it forces the hardware to take three times longer, magnifying the environmental error floor by a scale factor of $\lambda = 3$.
* Curve Fitting: We record our target double-slit signal at baseline hardware noise ($\lambda=1$, and amplified noise levels ($\lambda=3, \lambda=5$).
* Extrapolation: We fit a trend line (linear, polynomial, or exponential) across those noisy points, and mathematically trace it backward to where \$lambda = 0$. That intersection point represents your clean, error-mitigated quantum calculation.
## High-Fidelity ZNE Python Implementation
This script simulates a true hardware calibration pass. It establishes a baseline noise model, uses unitary gate folding to sample the system at amplified noise multipliers, and performs a Richardson Extrapolation to recover the pristine wavefront.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from braket.devices import LocalSimulator
from braket.circuits import Circuit, Gate
from braket.circuits.noise_model import NoiseModel, GateCriteria
from braket.circuits.noises import Depolarizing

# 1. Establish Experiment Topology and Base Noise Parameters
device = LocalSimulator("braket_dm")
phases = np.linspace(0, np.pi/2, 15)  # Zoomed fringe view

def get_noisy_distribution(base_circuit, noise_scale_factor):
    """
    Simulates scaling physical noise by adjusting error insertion rates.
    λ = 1.0 represents standard hardware baseline noise.
    """
    model = NoiseModel()
    criteria = GateCriteria(gates=[Gate.Rx, Gate.Rz])

    # Scale base hardware error (5% depolarizing) by our multiplier λ
    scaled_error_rate = 0.05 * noise_scale_factor
    model.add_noise(Depolarizing(scaled_error_rate), criteria)

    noisy_circ = model.apply(base_circuit)
    res = device.run(noisy_circ, shots=10000).result()
    return res.measurement_counts.get('0', 0) / 10000

# Arrays to hold results across our different hardware noise states
unmitigated_fringe = []    # λ = 1
amplified_noise_3x = []    # λ = 3
amplified_noise_5x = []    # λ = 5
mitigated_fringe = []      # Extrapolated λ = 0

# 2. Run the Double-Slit Sweep
for p in phases:
    # Compile the native hardware pulse sequence
    native_circuit = Circuit().rx(0, np.pi/2).rz(0, p).rx(0, -np.pi/2)

    # Sample the data at distinct noise levels (λ scale parameters)
    p_lambda_1 = get_noisy_distribution(native_circuit, noise_scale_factor=1.0)
    p_lambda_3 = get_noisy_distribution(native_circuit, noise_scale_factor=3.0)
    p_lambda_5 = get_noisy_distribution(native_circuit, noise_scale_factor=5.0)

    unmitigated_fringe.append(p_lambda_1)
    amplified_noise_3x.append(p_lambda_3)
    amplified_noise_5x.append(p_lambda_5)

    # 3. Perform Linear/Polynomial Richardson Extrapolation to solve for λ = 0
    # Formula uses a combination of data points to cancel out leading order error terms
    p_mitigated = (15/8 * p_lambda_1) - (5/4 * p_lambda_3) + (3/8 * p_lambda_5)

    # Clamp probability boundary to valid physical range [0, 1]
    mitigated_fringe.append(np.clip(p_mitigated, 0.0, 1.0))

# 4. Generate Ideal Reference Curve
ideal_fringe = []
for p in phases:
    circ_ideal = Circuit().h(0).rz(0, p).h(0)
    res_i = device.run(circ_ideal, shots=10000).result()
    ideal_fringe.append(res_i.measurement_counts.get('0', 0) / 10000)

# 5. Visual Verification Plot
plt.figure(figsize=(11, 6))
plt.plot(phases, ideal_fringe, label='Ideal Wavefront (No Noise)', color='black', linewidth=2.5)
plt.plot(phases, unmitigated_fringe, label='Raw QPU Data (λ=1 Baseline Noise)', color='crimson', marker='o')
plt.plot(phases, amplified_noise_3x, label='Amplified Noise (λ=3)', color='orange', linestyle=':', alpha=0.7)
plt.plot(phases, amplified_noise_5x, label='Amplified Noise (λ=5)', color='gold', linestyle=':', alpha=0.7)
plt.plot(phases, mitigated_fringe, label='🎉 QEM Mitigated Result (Extrapolated λ=0)', color='teal', linestyle='--', linewidth=2, marker='*')

plt.xlabel('Phase Shift (Radians)')
plt.ylabel('Probability |0> (Fringe Intensity)')
plt.title('Zero-Noise Extrapolation (ZNE) Error Mitigation Profile')
plt.grid(True, alpha=0.3)
plt.legend(loc='lower left')
plt.show()


## What to Look for in the Results
* The Degradation Slopes: Look at the step-downs from the Raw QPU Data ($\lambda=1$) to the Amplified Noise Lines ($\lambda=3$ and $\lambda=5$). As noise scales upward, the system bleeds coherence and the interference pattern flattens out.
* The Mathematical Correction: By running a Richardson polynomial fit through those values, the code calculates the trajectory of the error decay and shifts the points upward.
* The Visual Result: The Teal Star Line ($\lambda=0$) pulls itself back up to match the pristine black ideal wavefront line, effectively neutralizing the hardware's 5% gate depolarizing error using software post-processing alone.